## Sequence Purchasing Problem

Lead     : `<Alex / AlexLeonardos>`

Issue    : [Github Issue #84](https://github.com/petadex/igem-toronto/issues/) — _Sequence Purchasing Algorithm_

Start    : `2026-06-01`


## Background and Motivation

We want to optimize how we purchase DNA sequences by considering the value of Degenerate-Codon Oligo Libraries. We can purchase a sequence where at specific positions, different nucleotide bases can occur at a ratio we define. Using this, we could encode for many similar sequences from a desired cluster even though we only purchased 1 sequence. Therefore, our motivation is to find the best sequence(s) to purchase for our needs. 

There are several factors that we must account for:

1. **Sequence Information Quantity**: We have a limit in terms of how much information can be ordered, not necessarily the number of sequences. This is only quantified on the total sequences ordered over many clusters (libraries), so it's not computed for this algorithm, which outputs the information used in the optimal solution for specific input clusters (libraries).

2. **Sequence Degeneracy**: How many degenerate bases we will allow in any given sequence.

    - Want to *minimize junk*, critical part of Artem's specification.

3. **Possible Fragment Synthesis Approaches**: Whether ordering multiple smaller sequences and ligating them together could work. This idea is expanded upon in the *Important Operations section*.

4. **Sequence Coverage**: How many natural amino acid sequences (exist in the chosen families) these purchased DNA sequences cover.

5. **Codon Ratios**: Which ratio of bases at these positions would maximize the amount of natural sequences synthesized and minimize the amount of artificial ones that aren't biologically existing.

    - Computed using Lisa's *scoring function*. This considers the codon table for specific vectors which assays would take place using.
    - Max 4 custom ratio bases according to Twist, although we may be able to pay for more
        - IUPAC Standard 50/50 combinations don't count towards this custom ratio count

## Problem Statement

We want to create an algorithm that designs the best nucleotide sequence(s) to order for a given library, with a *junk ratio cap* for how much % junk is acceptable.

### Input: 

   1. **FASTA file for Input**: which consists of *aligned, trimmed protein sequences* that represent unique cores extracted from PETase clusters. The n_k at the end of the lines signifies how many natural sequences mapped to this core.  The fact that they are protein sequences is very important, as the algorithm must have a mechanism of *converting from nucleotides to amino acids*. 
Stored in the form:
```
>core1_n3
AAFAAPAGQTNPYARGPNPTAASLEASAGPFTVRSFTVSRPSGYGAGTVYYPTNAGGTVG
AIAIVPGYTARQSSIKWWGPRLASHGFVVITIDTNSTFDYPSSRSSQQMAALRQVASLNG
DSSSPIYGKVDTARMGVMGHSMGGGASLRSAANNPSLKAAIPQAPWDSQTNFSSVTVPTL
IFACENDSIAPVNSHALPIYDSMSRNAKQFLEINGGSHSCANSGNSNQALIGKKGVAWMK
RFMDNDTRYSTFACENPNSTAVSDFRTANCS-
>core2_n2
AVSAAATAQTNPYARGPNPTAASLEASAGPFTVRSFTVSRPSGYGAGTVYYPTNAGGTVG
AIAIVPGYTARQSSIKWWGPRLASHGFVVITIDTNSTLDQPSSRSSQQMAALRQVASLNG
TSSSPIYGKVDTARMGVMGWSMGGGGSLISAANNPSLKAAAPQAPWDSSTNFSSVTVPTL
IFACENDSIAPVNSSALPIYDSMSRNAKQFLEINGGSHSCANSGNSNQALIGKKGVAWMK
RFMDNDTRYSTFACENPNSTRVSDFRTANCS-
```
   2. **Junk Ratio Cap**: A scalar value in (0, 1) to determine what the maximum % of junk we would like to allow. This means that the algorithm must have a *manner of keeping track of how many possible proteins could be encoded for by the current solution*, which can be used to compute the junk ratio (how many aren't targets).
   3. **Fragment Joining Method**: This is either Golden Gate or Homologous Recombination. This defines what the junctions would be defined as. Note that due to these techniques and general plasmid design we are under the following hard constraints:

      1. **CGTCTC** and **CGTCTC** are BANNED from occuring ANYWHERE in the sequence.
      2. no internal CGGA or GGTG overhangs unless ur doing shared-overhang-minimal-plasmid stuff and want to exclude the first/last fragment(the backbone uses those two overhangs) 


### Output:

   1. The sequence to order - this is in nucleotide bases.
   2. Indication of which positions have degen codons, fragments, and how much junk this generates.

# Important Operations

These are the operations that can be used to design a sequence. The algorithm should consider these in some form when designing the final sequence over an input library.

1. **Degenerate Codons**: When at a certain index, there are several possibilities for nucleotide bases.
2. **Fragments**: Multiple shorter fragments can be ligated together by specific junctions. This allows for consideration of library sequences of different lengths, or for protein differences that can't be efficiently encoded for using degenerate codons.

# Key Definitions:

Let $L_T$ be our target library of cores. This is a weighted set, with each $s \in L_T$ having a weight $w(s)$. 

*Justification for $w(s)$*: In preprocessing before the algorithm, many of the ORFs contain the exact same core sequence. The weight is defined as the number of natural ORFs that are behind the core $s$. Ths allows for maximization of natural sequence count, but does not change the number of unique cores ordered. Instead, it allows us to choose which cores to include. This is justifiable, as a core observed in more ORFs is more likely to be valid and not an assembly artifact/outlier.

Let $L_O$ be the protein library encoded by our algorithm's output sequences. This is a cartesian product across fragments and degenerate codons, meaning that it grows quickly with the number of variations added.

1. **Coverage**: A core $s \in L_T$ is *covered* if $s \in L_O$. 
   Therefore, the total coverage is defined as $\frac{\sum_{s \in L_O \cap L_T} w(s)}{\sum_{s \in L_T} w(s)}$. This is weighted by $w(s)$.

2. **Junk**: This loosely means useless sequences. This can be split into two different type of junk, minimized at different points in the algorithm.

   a. **Counting Junk**: This is $1 - \frac{|L_T \cap L_O|}{|L_O|}$. In English, this is the proportion of our output library that's *not* covering a sequence in the target library. This is the junk that represent the cap in the initial stage of the algorithm. Note that this junk cap is not a quality metric, it is a feasibility constraint. It is also not weighted, as the entries of $L_O$ do not have inherent weights, which are relevant for coverage.

   b. **Mass Junk**: This is the probability mass of landing on non-targets. This can be influenced by *custom degenerate codon ratios*, which are a future direction for optimizing in specific species. Custom ratios cannot change $L_O$, only the distribution over it — so mass junk is optimized on the support fixed by stage 1, leaving coverage and counting junk unchanged. Note that counting junk is a special case of mass junk, under the assumption that the output library is uniformly distributed.

## For Future Work:

These are ideas that would be nice to have in a complete, publishable version of this algorithm. For this first pass of sequence purchasing, they do not need to be explicitly considered.

1. Structural Modelling - Accounting for if RNA structures (hairpins) make the sequence infeasible. This was previously a main feature of the algorithm's design, but has since been removed.
2. Proofs about Approximation Ratio?
3. **Custom Ratios**: Which ratio of bases at these positions would maximize the amount of natural sequences synthesized and minimize the amount of artificial ones that aren't biologically existing.

# Important Note: DARWIN Algorithm Implementation

Oscar has described the DARWIN algorithm for finding the top k most influential mutations informed by existing activity data. This will likely be used to generate the first set of sequences for purchasing. Therefore a much simpler version of this sequence purchasing problem will be implemented if the DARWIN algorithm is effective.

This algorithm will simply incorporate the top K mutations, adding the proper degenerate codons for substitution mutations, and fragmentation for addition/deletion mutations.